In [36]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



In [37]:
'''import shutil
import os

# This clears the old 384-dim database
db_path = "../data/vector_store" 
if os.path.exists(db_path):
    shutil.rmtree(db_path)
    print("✅ Old database deleted. Ready for high-precision 1024-dim model.")'''

'import shutil\nimport os\n\n# This clears the old 384-dim database\ndb_path = "../data/vector_store" \nif os.path.exists(db_path):\n    shutil.rmtree(db_path)\n    print("✅ Old database deleted. Ready for high-precision 1024-dim model.")'

In [38]:
import os
from pathlib import Path

# Use Path.home() to safely grab /Users/anudeepmanda/ regardless of OS
# Then append the folder name 'resumes'
resume_folder_path = os.path.join(Path.home(), "Downloads" , "Resume")

def process_resumes(folder_path):
    all_docs = []
    
    # Check if the folder exists before trying to read it
    if not os.path.exists(folder_path):
        print(f"Error: The folder {folder_path} does not exist. Please check the name.")
        return []

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)
            
            # Load only relevant document types
            if file.endswith(".pdf"):
                loader = PyMuPDFLoader(file_path)
            elif file.endswith(".docx"):
                loader = Docx2txtLoader(file_path)
            else:
                continue
                
            try:
                loaded_docs = loader.load()
                # Add metadata tags to help the RAG prioritize these
                for doc in loaded_docs:
                    doc.metadata["doc_type"] = "resume"
                    doc.metadata["file_name"] = file
                all_docs.extend(loaded_docs)
            except Exception as e:
                print(f"Could not load {file}: {e}")
                
    return all_docs

# Run the updated loader
all_resume_documents = process_resumes(resume_folder_path)
print(f"Successfully loaded {len(all_resume_documents)} documents from {resume_folder_path}")

Could not load ~$udeep_manda_ramp.docx: File is not a zip file
Successfully loaded 89 documents from /Users/anudeepmanda/Downloads/Resume


In [39]:

all_resume_documents

[Document(metadata={'source': '/Users/anudeepmanda/Downloads/Resume/anudeep_manda_mongodb_is.docx', 'doc_type': 'resume', 'file_name': 'anudeep_manda_mongodb_is.docx'}, page_content='Anudeep Manda    \n\nmanda1@purdue.edu | 925-683-7968 | linkedin.com/in/anudeepmanda   \n\nEDUCATION\n\n\t\t\t\t\t\t\tPurdue University – West Lafayette, IN       \t  \t  \t  \t  \t  \t  \t       GPA: 3.8   May 2027  \n\n\t\t\t\t\tB.S. in Computer Engineering and Applied Mathematics (Double Major)  \t     \t  \t  \t  \t               \n\nCoursework: Advanced C Programming, Object-Oriented Programming (C++), Data Structures & Algorithms, Operating Systems,  Full-Stack Web Development (Microsoft Learn), Cloud Computing (AWS Sagemaker)\n\nTECHNICAL SKILLS\n\nLanguages: Java, Python, JavaScript/TypeScript, SQL, C++\n\nApplication Development: RESTful APIs, Node.js, React.js, Object-Oriented Design, Microservices concepts, Agile/Scrum\n\nDatabases: MongoDB, PostgreSQL, MySQL, Firebase, Data modeling, query opti

In [40]:
def split_documents(document):
    custom_seperators = [
        "\nEDUCATION", 
        "\nTECHNICAL SKILLS",
        "\nEXPERIENCE", 
        "\nPROJECTS",
        "\n\n", 
        "\n", 
        "•", 
        " "
    ]
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1500,
        chunk_overlap = 150,
        length_function = len, 
        separators=  custom_seperators, 
        is_separator_regex= False
    )
    split_docs = text_splitter.split_documents(document)
    print(f"Split {len(document)} documents into {len(split_docs)} chunks")
    

    return split_docs


In [41]:
#chunks_pdf = split_documents(all_pdf_documents)
chunks_docx = split_documents(all_resume_documents)
#print(chunks_pdf)
print(chunks_docx)


Split 89 documents into 514 chunks
[Document(metadata={'source': '/Users/anudeepmanda/Downloads/Resume/anudeep_manda_mongodb_is.docx', 'doc_type': 'resume', 'file_name': 'anudeep_manda_mongodb_is.docx'}, page_content='Anudeep Manda    \n\nmanda1@purdue.edu | 925-683-7968 | linkedin.com/in/anudeepmanda'), Document(metadata={'source': '/Users/anudeepmanda/Downloads/Resume/anudeep_manda_mongodb_is.docx', 'doc_type': 'resume', 'file_name': 'anudeep_manda_mongodb_is.docx'}, page_content='EDUCATION\n\n\t\t\t\t\t\t\tPurdue University – West Lafayette, IN       \t  \t  \t  \t  \t  \t  \t       GPA: 3.8   May 2027  \n\n\t\t\t\t\tB.S. in Computer Engineering and Applied Mathematics (Double Major)  \t     \t  \t  \t  \t               \n\nCoursework: Advanced C Programming, Object-Oriented Programming (C++), Data Structures & Algorithms, Operating Systems,  Full-Stack Web Development (Microsoft Learn), Cloud Computing (AWS Sagemaker)'), Document(metadata={'source': '/Users/anudeepmanda/Downloads/R

In [42]:
%pip install flashrank

Note: you may need to restart the kernel to use updated packages.


In [43]:
from flashrank import Ranker, RerankRequest

ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2", cache_dir="/tmp")

In [44]:
# Embedding and Vector Store

In [45]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [46]:
class EmbeddingManager:
    #handles domcument embedding gneration using SentenceTransfomer

    def __init__(self, model_name: str = 'BAAI/bge-large-en-v1.5'):
        '''
        Initialie the embedding manager
        model_name -> HuggingFace model name for sentences embeddings

        '''
        self.model_name = model_name
        self.model = None
        self._load_model() #loads model 

    def _load_model(self): #protected funciton 
        try:
            print(f"Loading embdding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded succesfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") # every text converted into dimensions
        except Exception as e:
            print(f"Error loading model {self.model}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]):
        if not self.model: 
            raise ValueError("Model not loaded")
        print(f"Generate Embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar= True)
        print(f"Generate embeddings with shape: {embeddings.shape}")
        return embeddings


##initializze the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


Loading embdding model : BAAI/bge-large-en-v1.5


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1

Model loaded succesfully. Embedding dimension: 1024


In [47]:
#vector store

In [48]:
import numpy as np
import os
import hashlib
from typing import List, Any


class VectorStore:
    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try: 
            #create persistent chromadb client 
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #create or get collection 
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}

            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e: 
            print(f"Error intializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match with the number of embeddings")

        print(f"Adding/Updating {len(documents)} documents in the vector store...")

        ids = []
        metadata_list = []
        documents_text = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # 1. Get the filename (essential for differentiating similar resumes)
            # Using .get() ensures it doesn't crash if the key is missing
            file_name = doc.metadata.get('file_name') or doc.metadata.get('source') or "unknown"
            file_name = os.path.basename(file_name) # Clean the path to just the name

            # 2. Create a hash of the content to detect if the text has changed
            content_hash = hashlib.md5(doc.page_content.encode()).hexdigest()

            # 3. Create a Deterministic ID
            # Format: filename_index_hash
            # This keeps 'Anudeep_Security' and 'Anudeep_ML' separate even if the text is similar
            doc_id = f"{file_name}_{i}_{content_hash[:10]}"
            ids.append(doc_id)

            # 4. Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i 
            metadata['content_length'] = len(doc.page_content)
            metadata['file_name'] = file_name # Standardize the filename key
            metadata_list.append(metadata)

            documents_text.append(doc.page_content)

        # 5. Use 'upsert' instead of 'add'
        # 'upsert' will update existing IDs and add new ones, preventing duplicates
        try:
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings.tolist(),
                metadatas=metadata_list,
                documents=documents_text
            )
            print(f"✅ Successfully synced {len(documents)} chunks.")
            print(f"Total unique chunks in collection: {self.collection.count()}")
        except Exception as e: 
            print(f"Error syncing chunks to vector store: {e}")
            raise

    def clear_collection(self):
        try: 
            self.client.delete_collection(name = self.collection_name)
            print("Successfully cleared the collection")
            self._initialize_store()

        except Exception as e: 
            print(f"Error clearing the collection : {e}")
            raise
vectorstore = VectorStore()
vectorstore




Vector store initialized. Collection: pdf_documents
Existing documents in collection: 514


In [49]:
#convert chunks texts in embeddings

#texts_pdf = [doc.page_content for doc in chunks_pdf]
texts_docx = [doc.page_content for doc in chunks_docx]

#generate the embeddings

#embeddings_pdf = embedding_manager.generate_embeddings(texts_pdf)
embeddings_docx = embedding_manager.generate_embeddings(texts_docx)

#store in the vector database
#vectorstore.add_documents(chunks_pdf, embeddings_pdf)
vectorstore.add_documents(chunks_docx, embeddings_docx)

# how to remove duplicates?


Generate Embeddings for 514 texts...


Batches: 100%|██████████| 17/17 [00:17<00:00,  1.05s/it]

Generate embeddings with shape: (514, 1024)
Adding/Updating 514 documents in the vector store...
Error syncing chunks to vector store: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database


InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database

In [ ]:
#chunks_pdf

In [ ]:
from flashrank import Ranker, RerankRequest

class RagRetriever:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        # Initialize the high-speed re-ranker
        self.ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2", cache_dir="/tmp")
    
    def find_best_document(self, job_description: str, fetch_k: int = 25):
        print(f"--- Phase 1: Initial Retrieval (Fetching {fetch_k} chunks) ---")
        query = job_description.strip()
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # 1. Broad Search: Get more chunks than you need (e.g., 25)
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()], 
            n_results=fetch_k
        )

        ids = results['ids'][0]
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]

        # 2. Format for Re-Ranker
        # Note: Make sure 'file_name' matches the metadata key you used in Step 1
        passages = []
        for doc_id, text, meta in zip(ids, documents, metadatas):
            passages.append({
                "id": doc_id,
                "text": text,
                "meta": meta
            })

        print(f"--- Phase 2: Deep Re-Ranking (Cross-Encoding) ---")
        rerank_request = RerankRequest(query=query, passages=passages)
        reranked_results = self.ranker.rerank(rerank_request)

        # 3. Aggregate results by File Name
        leaderboard = {}
        for res in reranked_results:
            filename = res['meta'].get('file_name', 'Unknown File') # Ensure this matches your metadata key
            score = res['score']
            
            if filename not in leaderboard:
                leaderboard[filename] = {
                    'max_score': 0.0,
                    'hit_count': 0,
                    'top_text': res['text']
                }
            
            # Using Max Score is often better than Total Score for resumes
            if score > leaderboard[filename]['max_score']:
                leaderboard[filename]['max_score'] = score
                leaderboard[filename]['top_text'] = res['text']
            
            leaderboard[filename]['hit_count'] += 1

        # 4. Display Results
        ranked_files = sorted(leaderboard.items(), key=lambda x: x[1]['max_score'], reverse=True)

        for rank, (filename, stats) in enumerate(ranked_files, 1):
            print(f"\nRANK #{rank}: {filename}")
            print(f"Confidence Score: {stats['max_score']:.4f}")
            print(f"Key Match: {stats['top_text'][:200]}...")

        return ranked_files
    
    
    

# Re-initialize with the new logic
rag_retriever = RagRetriever(vectorstore, embedding_manager)

In [ ]:
rag_retriever

In [ ]:
# 1. Define a specific, high-signal Job Description
# This matches many of the themes in your cover letter (OpenAI, Python, Docker)
test_job_description = """
Undergrad Software Engineering Intern
at Twitch
San Francisco, CA

25%
Resume Match
1 of 4 keywords
Simplify
Simplify
V1
V2
Please apply to this job with your personal email address (not your University or Program account) to avoid important Twitch emails hitting your spam inbox.

About Us:

Launched in 2011, Twitch is a global community that comes together each day to create multiplayer entertainment: unique, live, unpredictable experiences created by the interactions of millions. We bring the joy of co-op to everything, from casual gaming to world-class esports to anime marathons, music, and art streams. Twitch also hosts TwitchCon, where we bring everyone together to celebrate, learn, and grow their personal interests and passions. We’re always live at Twitch. Stay up to date on all things Twitch on Linkedin, X and on our Blog.

About the Role

**This internship is only for undergraduate students at North American colleges/universities with Winter 2026 or Spring 2027 graduation dates**

At Twitch, we’re always looking for high-potential talent. If you’re a current student in computer science, we’d love to see you apply. See below for just a few reasons why our cohort-based internship program is one-of-a kind. Project-Based Learning at Scale: Twitch operates at a massive scale, requiring us to push the boundaries of technology and experiment with techniques used by only the largest websites. As an intern, you’ll have the opportunity to work on some of the most challenging engineering problems in the industry, making every project a valuable learning experience. 

Comprehensive Support Team: To ensure our interns thrive, we provide a dedicated support system, including a Manager, Mentor, and Early Careers Advisor. This team will work closely with you to guide your project from start to finish and ensure you have the resources needed to succeed. 

Enriched Intern Community: In addition to hands-on engineering work, you will participate in leadership and interpersonal development curriculum, as well as gain valuable industry insights from Twitch leaders through fireside chats. To complement the technical aspects of the program, we offer a variety of enrichment activities, including cohort trips, weekly advisory sessions, and local community events. We also provide housing assistance to make your summer internship as rewarding and enjoyable as possible.

Want to learn more? Check out the Early Careers Page for internship and student-focused content. 

You can work from San Francisco, CA.

You Will:

Contribute to product design and implementation discussions
Find and build unique solutions to implement projects from the idea phase to production
Test and iterate code before and after production release
Connect with intern peers, new teammates, and colleagues from across the business
You Have:

BA/BS Graduation Year: Winter 2026 or Spring/Summer 2027. No Masters/PhD
Eagerness to investigate challenges and present reasonable solutions reliably and quickly 
Excitement to pick up a new language and be productive with it in a week 
Desire to work collaboratively in a team environment 
Completed coursework in or have an understanding of Data Structures and Algorithms


"""

# 2. Execute the Search
# We fetch 20 chunks initially to give the Re-Ranker enough "candidates" to evaluate
print("🔍 Searching for the top candidates...")
ranked_results = rag_retriever.find_best_document(test_job_description, fetch_k=20)

# 3. Clean Results Display
print("\n" + "="*60)
print("🏆 FINAL RANKINGS (The 'Best Fit' List)")
print("="*60)

if ranked_results:
    # We only display the top 5 to focus on the highest quality matches
    for rank, (filename, stats) in enumerate(ranked_results[:5], 1):
        score = stats['max_score']
        
        # Determine "Match Quality" for quick visual feedback
        quality = "🟢 High Match" if score > 0.7 else "🟡 Moderate Match" if score > 0.4 else "🔴 Low Match"
        
        print(f"RANK #{rank}: {filename}")
        print(f"   Status: {quality} (Score: {score:.4f})")
        print(f"   Key Matching Experience: {stats['top_text'][:200]}...")
        print("-" * 60)
else:
    print("❌ No matches found. Check your 'resumes' folder path or vectorstore initialization.")

🔍 Searching for the top candidates...
--- Phase 1: Initial Retrieval (Fetching 20 chunks) ---
Generate Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]


Generate embeddings with shape: (1, 1024)
--- Phase 2: Deep Re-Ranking (Cross-Encoding) ---

RANK #1: anudeep_manda_nuro.docx
Confidence Score: 0.2057
Key Match: EXPERIENCE  

RTX | Collins Aerospace 

Software Engineering Intern 								                         Jan 2026 – Present

Developing and validating low-level embedded firmware and drivers C/C++ for avi...

RANK #2: anudeep_manda_anduril.docx
Confidence Score: 0.1842
Key Match: EXPERIENCE  

RTX | Collins Aerospace 

Software Engineering Intern 								                         Jan 2026 – Present

Developing and validating low-level embedded firmware and drivers C/C++ for avi...

RANK #3: anudeep_manda_rivian_swe.docx
Confidence Score: 0.1716
Key Match: EXPERIENCE  

RTX | Collins Aerospace 

Software Engineering Intern 								                         Jan 2026 – Present

Developing and validating low-level embedded firmware and drivers C/C++ for avi...

RANK #4: anudeep_manda_amez.docx
Confidence Score: 0.1639
Key Match: Softwar

In [ ]:
# Check current database content
'''current_docs = vectorstore.collection.get()
unique_files = set(meta.get('file_name') or meta.get('source') for meta in current_docs['metadatas'])

print(f"The database currently contains {len(current_docs['ids'])} chunks.")
print("It is still remembering these files:")
for f in unique_files:
    print(f" - {f}")
'''

'current_docs = vectorstore.collection.get()\nunique_files = set(meta.get(\'file_name\') or meta.get(\'source\') for meta in current_docs[\'metadatas\'])\n\nprint(f"The database currently contains {len(current_docs[\'ids\'])} chunks.")\nprint("It is still remembering these files:")\nfor f in unique_files:\n    print(f" - {f}")\n'